In [0]:
import streamlit as st
import pandas as pd

from databricks.sdk import WorkspaceClient

from datetime import datetime, timezone


# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Databricks Job Monitor",
    page_icon="📊",
    layout="wide"
)


# =========================================================
# DATABRICKS CLIENT
# =========================================================

@st.cache_resource
def get_databricks_client():
    return WorkspaceClient()


try:
    w = get_databricks_client()

except Exception as e:

    st.error("Unable to create Databricks client.")

    st.exception(e)

    st.stop()


# =========================================================
# AUTHENTICATION
# =========================================================

try:

    current_user = w.current_user.me()

except Exception as e:

    st.error("Databricks authentication failed.")

    st.exception(e)

    st.stop()


# =========================================================
# HEADER
# =========================================================

st.title("📊 Databricks Job Monitor")

st.caption(
    f"Authenticated as: {current_user.user_name}"
)


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def format_timestamp(timestamp):

    if timestamp is None:

        return "-"

    try:

        dt = datetime.fromtimestamp(
            timestamp / 1000,
            tz=timezone.utc
        )

        return dt.strftime(
            "%Y-%m-%d %H:%M:%S UTC"
        )

    except Exception:

        return "-"


def get_job_name(job):

    try:

        if job.settings and job.settings.name:

            return job.settings.name

    except Exception:

        pass

    return f"Job {job.job_id}"


def get_job_creator(job):

    try:

        if job.creator_user_name:

            return job.creator_user_name

    except Exception:

        pass

    return "-"


# =========================================================
# GET JOBS
# =========================================================

@st.cache_data(ttl=60)
def get_jobs():

    jobs = []

    try:

        response = w.jobs.list()

        for job in response:

            if job.job_id is None:

                continue

            jobs.append(
                {
                    "job_id":
                        job.job_id,

                    "job_name":
                        get_job_name(job),

                    "created_time":
                        format_timestamp(
                            job.created_time
                        ),

                    "updated_time":
                        format_timestamp(
                            job.change_time
                        ),

                    "created_by":
                        get_job_creator(job)
                }
            )

        return jobs


    except Exception as e:

        return {
            "error": str(e)
        }


# =========================================================
# GET JOB RUNS
# =========================================================

@st.cache_data(ttl=60)
def get_job_runs(job_id):

    try:

        response = w.jobs.list_runs(
            job_id=job_id,
            limit=25
        )

        return list(response)

    except Exception:

        return []


# =========================================================
# LOAD JOBS
# =========================================================

jobs_result = get_jobs()


# =========================================================
# HANDLE ERROR
# =========================================================

if isinstance(jobs_result, dict):

    st.error(
        "Unable to retrieve Databricks jobs."
    )

    st.code(
        jobs_result.get(
            "error",
            "Unknown error"
        )
    )

    st.warning(
        """
        The Streamlit App identity may not have
        permission to view Databricks jobs.
        """
    )

    st.stop()


jobs = jobs_result


# =========================================================
# DEBUG INFORMATION
# =========================================================

st.success(
    f"Successfully retrieved {len(jobs)} Databricks job(s)."
)


# =========================================================
# REFRESH
# =========================================================

if st.button("🔄 Refresh"):

    st.cache_data.clear()

    st.rerun()


# =========================================================
# FILTERS
# =========================================================

st.subheader("🔎 Filters")


filter_col1, filter_col2 = st.columns(2)


# ---------------------------------------------------------
# JOB NAME
# ---------------------------------------------------------

job_names = sorted(
    list(
        set(
            job["job_name"]
            for job in jobs
        )
    )
)


with filter_col1:

    selected_jobs = st.multiselect(
        "Job Name",
        options=job_names,
        placeholder="Select jobs"
    )


# ---------------------------------------------------------
# CREATED BY
# ---------------------------------------------------------

created_by_values = sorted(
    list(
        set(
            job["created_by"]
            for job in jobs
            if job["created_by"] != "-"
        )
    )
)


with filter_col2:

    selected_user = st.selectbox(
        "Created By",
        ["All Users"] + created_by_values
    )


# =========================================================
# APPLY FILTERS
# =========================================================

filtered_jobs = jobs.copy()


if selected_jobs:

    filtered_jobs = [
        job
        for job in filtered_jobs
        if job["job_name"]
        in selected_jobs
    ]


if selected_user != "All Users":

    filtered_jobs = [
        job
        for job in filtered_jobs
        if job["created_by"]
        == selected_user
    ]


st.info(
    f"Showing {len(filtered_jobs)} of {len(jobs)} jobs"
)


# =========================================================
# TABLE 1
# JOB INFORMATION
# =========================================================

st.subheader(
    "1️⃣ Job Information"
)


job_information = []


for job in filtered_jobs:

    job_information.append(
        {
            "Workspace Name":
                "Databricks Workspace",

            "Job Name":
                job["job_name"],

            "Job ID":
                job["job_id"],

            "Created Date":
                job["created_time"],

            "Last Update Date":
                job["updated_time"],

            "Created By":
                job["created_by"]
        }
    )


if len(job_information) > 0:

    job_information_df = pd.DataFrame(
        job_information
    )

    st.dataframe(
        job_information_df,
        use_container_width=True,
        hide_index=True
    )

else:

    st.warning(
        "No jobs match the selected filters."
    )


# =========================================================
# TABLE 2
# JOB RUN SUMMARY
# =========================================================

st.subheader(
    "2️⃣ Job Run Summary"
)


run_summary = []


for job in filtered_jobs:

    job_id = job["job_id"]

    job_name = job["job_name"]


    runs = get_job_runs(
        job_id
    )


    total_runs = len(
        runs
    )

    success_runs = 0

    failed_runs = 0


    for run in runs:

        try:

            if not run.state:

                continue


            status = "UNKNOWN"


            if run.state.result_state:

                status = (
                    run.state
                    .result_state
                    .value
                )


            elif run.state.life_cycle_state:

                status = (
                    run.state
                    .life_cycle_state
                    .value
                )


            status = status.upper()


            if status in [
                "SUCCESS",
                "SUCCEEDED"
            ]:

                success_runs += 1


            elif status in [
                "FAILED",
                "ERROR",
                "TIMED_OUT"
            ]:

                failed_runs += 1


        except Exception:

            continue


    completed_runs = (
        success_runs +
        failed_runs
    )


    if completed_runs > 0:

        success_ratio = (
            success_runs /
            completed_runs
        ) * 100

    else:

        success_ratio = 0


    run_summary.append(
        {
            "Job Name":
                job_name,

            "Job ID":
                job_id,

            "Total Runs":
                total_runs,

            "Success Runs":
                success_runs,

            "Failed Runs":
                failed_runs,

            "Success Ratio":
                f"{success_ratio:.2f}%"
        }
    )


if len(run_summary) > 0:

    run_summary_df = pd.DataFrame(
        run_summary
    )

    st.dataframe(
        run_summary_df,
        use_container_width=True,
        hide_index=True
    )

else:

    st.warning(
        "No job run information available."
    )


# =========================================================
# SUMMARY METRICS
# =========================================================

st.divider()

total_jobs = len(
    filtered_jobs
)

total_runs = sum(
    row["Total Runs"]
    for row in run_summary
)

successful_runs = sum(
    row["Success Runs"]
    for row in run_summary
)

failed_runs = sum(
    row["Failed Runs"]
    for row in run_summary
)


col1, col2, col3, col4 = st.columns(4)


with col1:

    st.metric(
        "Total Jobs",
        total_jobs
    )


with col2:

    st.metric(
        "Total Runs",
        total_runs
    )


with col3:

    st.metric(
        "Successful Runs",
        successful_runs
    )


with col4:

    st.metric(
        "Failed Runs",
        failed_runs
    )


# =========================================================
# FOOTER
# =========================================================

st.divider()

st.caption(
    "Databricks Job Monitor"
)